# PosgenPy Grid Analysis (Python-only)

A second workflow over the same engine as
`PosgenPy_Full_Analysis_Example_PythonOnly.ipynb`. That notebook sweeps one
parameter at a time - Order at a fixed dmax, then dmax at a fixed Order - and
reads each sweep on its own graph.

This one starts from the **whole (Order x dmax) plane**, because the accelerated
sweep made the plane cost about what one line used to: per dataset, one kNN
query and one radius query answer every cell, so 30 cells cost barely more than
6. Everything after that reads the map rather than a line.

1. **The map** - one `run_fast_grid_sweep` over both axes at once.
2. **Six heatmaps** - how many clusters the search finds, the `n_min` needed
   before 95% of them are real, how many survive it, how big the biggest gets,
   and how close the closest pair is (in nm, and in units of their own radii).
3. **Distributions over the plane** - cluster size and nearest-neighbour
   distance as ridgelines: one row per Order, one coloured half violin per
   dmax, with every cluster drawn as its own hoverable point.
4. **Splits and merges between neighbouring cells** - every cell clusters the
   *same atoms*, so a cluster can be followed from one cell to the next. The
   step that suddenly splits or merges a lot of clusters is the elbow.
5. **Individual clusters** - a `cluster_id` read off any plot above, in 3D, and
   the same cluster across a whole axis of the map to watch it split.
6. **The final run** at the chosen parameters - envelope, erosion and
   corrections - with its clusters drawn as **see-through alpha shapes**, edge
   clusters in their own colour.
7. **Composition** of those clusters - a Mn-Ni-Si ternary and one 100% column
   per cluster, smallest to largest - each beside the pooled composition of
   all clusters and the matrix.

**Tooltips are curated, not exhaustive.** Every plot leads with `cluster_id` and
then shows the handful of columns worth reading (size, neighbour distance, edge
flag) instead of the 27 a stats table carries. Pass
`hover=["cluster_id", "size_core", "Ni"]` to pick your own, or `hover="all"` for
the old everything-tooltip.

It runs top to bottom on the small bundled test dataset
(`posgenpyTest_easy.pos` + `examples/posgenpyTest.rrng`); swap the paths and ion
lists in the INPUT REQUIRED cells for your own data.

In [ ]:
######################### INPUT REQUIRED #########################
import numpy as np
import pandas as pd

import posgenpy as pgp

sample_name = "PosgenPyTest"
pos_file_path = "posgenpyTest_easy.pos"
range_file_path = "examples/posgenpyTest.rrng"
output_folder = "pygenpy_output_grid/"

core_ions = ["Cu", "Ni", "Si", "Mn"]
bulk_ions = ["Fe", "Mo", "C", "Cr", "P"]

random_runs = 30                 # randomised control datasets per grid cell
n_jobs = -1                      # control datasets at once (-1 = all cores)
confidence_threshold = 0.95      # the real-vs-random ratio n_min must reach
detector_efficiency = 0.52       # LEAP5000+ ~0.52, LEAP3000 ~0.37
atomic_density_per_nm3 = 85.49   # BCC iron
elements_to_exclude = ["Fe"]     # left out of the "cluster size" column

## Load data

In [ ]:
pos_df = pgp.read_pos_to_df(pos_file_path)
rrng_df = pgp.read_rrng(range_file_path)

# AP Suite exports don't use the "Range..." line format read_rrng expects -
# fall back to the AP-Suite reader if nothing was parsed.
if rrng_df.empty:
    rrng_df = pgp.read_apt_range_file(range_file_path)

ion_colors = pgp.get_ion_colours_from_rrng(rrng_df)

print(f"{len(pos_df):,} atoms")
rrng_df

## 1. The map

`run_fast_grid_sweep` runs the core-classification/linking half of the cluster
search at every combination of `orders` x `dmax_values`, for the real tip and
for `random_runs` relabelled control datasets, and writes the same per-cluster
parquet files the 1D sweeps write.

Per dataset it computes **once** the distance from every core-species atom to
its k-th nearest core-species neighbour (one kNN query at `k = max(order)`) and
every pair of core-species atoms closer than `max(dmax)` (one radius query). An
atom is a core point at `(order, dmax)` exactly when its order-th neighbour
distance is `< dmax`, and two core points link exactly when their pair distance
is `< dmax` - so every cell after the first is a re-threshold of arrays already
in memory. Cost scales with the number of **datasets**, not datasets x cells,
which is why the whole plane is affordable.

`progress=True` puts one bar over every cluster search the map runs - each
cell for the real tip and for every control dataset, `len(orders) x
len(dmax_values) x (random_runs + 1)` in all - with the latest message on the
line under it instead of a growing log. The real tip moves it a cell at a
time; a control dataset clusters every cell in one pass, so each one that
finishes moves it a whole row of cells. Cells already on disk from an
interrupted run count as done from the start.

Re-running this cell **resumes**: cells already in `output_folder` are reused,
but only ones made from the same inputs - the same atoms as core species (so
the same tip and `core_ions`), the same `n_min`. Change any of those and it
stops and says so rather than mixing two sweeps into one map; point
`output_folder` somewhere new, or pass `resume=False` to recompute it all.

`n_min=1` keeps every cluster the search found: the heatmaps below want the
unclipped count, and each cell's own n_min is chosen afterwards from the same
files.

Envelope and erosion are skipped here - they only ever add or remove *bulk*
atoms, so they cannot change the core+border counts this stage reads. They come
back for the final run in section 7, where composition and volume depend on
them.

In [ ]:
######################### INPUT REQUIRED #########################
orders = [3, 4, 5, 6, 7, 8, 9]
dmax_values = [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]

grid = pgp.run_fast_grid_sweep(
    pos_df, rrng_df, core_ions,
    orders=orders, dmax_values=dmax_values,
    bulk_ions=bulk_ions, n_min=1, n_relabel_runs=random_runs,
    output_dir=output_folder, prefix=sample_name,
    n_jobs=n_jobs,
    verbose=True,
    # one bar over all len(orders) x len(dmax_values) x (random_runs + 1)
    # cluster searches, and the latest message on the line under it
    progress=True,
)

print(f"\n{len(grid)} cells written to {output_folder}")

## 2. Reading the plane: six heatmaps

`summarise_grid` reduces every cell to one row. The n_min rule is
`optimise_n_min`'s, called once per cell, so there is one definition of it in
the package; the size and separation columns are measured over the clusters
that survive **that cell's own n_min**, since the ones below it are exactly the
noise the n_min was chosen to remove.

| heatmap | what it says |
|---|---|
| `n_clusters_all` | everything the search called a cluster, before n_min. At a loose (Order, dmax) most of it is noise, and this is the number that shows how much |
| `optimal_n_min` | the smallest n_min at which `(real - random)/real` reaches the confidence threshold. A cell where this is huge is one where the parameters are finding mostly random structure |
| `n_real_clusters` | how many clusters are left after that n_min - the actual yield of the cell |
| `max_cluster_size` | the largest surviving cluster. A cell where this jumps has started merging neighbouring precipitates into one |
| `min_cluster_distance` | the closest two cluster centres, in nm |
| `min_cluster_distance_norm` | the same distance in units of the two clusters' radii. **Below 1 their equivalent spheres overlap** - one precipitate reported as two, or two about to merge at the next dmax |

Read them together: the region worth choosing from is a **plateau**, where
`optimal_n_min` and `n_real_clusters` are both flat over neighbouring cells.
A cell that looks best but sits on a cliff is balanced on a knife edge, and the
CV01 check at the end will say so.

Each quantity gets its own colour scale on purpose - six identically-coloured
heatmaps of the same grid are impossible to tell apart at a glance. The
normalised distance gets a diverging one centred on 1, so "overlapping" and
"separated" land on opposite poles.

In [ ]:
summary = pgp.summarise_grid(
    grid, n_min_values=range(1, 200), confidence_threshold=confidence_threshold
)
summary

In [ ]:
heatmaps = [
    dict(value="n_clusters_all", colorscale="Blues",
         title="clusters found, before n_min"),
    dict(value="optimal_n_min", colorscale="Purples",
         title=f"n_min needed for {confidence_threshold:.0%} real clusters"),
    dict(value="n_real_clusters", colorscale="Greens",
         title="clusters surviving that n_min"),
    dict(value="max_cluster_size", colorscale="Oranges",
         title="largest surviving cluster [core+border atoms]"),
    dict(value="min_cluster_distance", colorscale="Teal", text_format=".2f",
         title="closest two cluster centres [nm]"),
    dict(value="min_cluster_distance_norm", colorscale="RdBu", zmid=1,
         text_format=".2f",
         title="closest pair / (r₁ + r₂)  -  below 1 they overlap"),
]

for spec in heatmaps:
    spec = dict(spec, title=f"{sample_name}: {spec['title']}")
    pgp.plot_grid_heatmap(summary, **spec).show()

## 3. Distributions over the plane

A violin per cell would be 30 violins nobody can read side by side. These are
**ridgelines** instead: one subplot row per Order, and inside each row one
filled density curve per dmax - a half violin on its side - so the figure reads
as "how does this distribution move as dmax rises (the coloured curves), and
does that change with Order (the rows)".

Every cluster is still drawn individually, as a point under its own curve, and
each point's tooltip leads with its `cluster_id`. That is the round trip: spot
an outlier in a tail here, read its id, and put it into section 6 to see it in
3D.

`load_grid_cluster_table` is what feeds them - every real cluster of every cell
as one row, clipped at that cell's optimal n_min, with the nearest-neighbour
distance measured **within its own cell** (a cluster's neighbours are different
at every parameter pair).

Swap `row=` and `color=` to read the map the other way round.

In [ ]:
grid_clusters = pgp.load_grid_cluster_table(grid, n_min=summary)

print(f"{len(grid_clusters):,} clusters across {len(grid)} cells")
grid_clusters.head()

In [ ]:
# cluster size: log x, because sizes span two decades
pgp.plot_distribution_grid(
    grid_clusters, value="size_core", row="order", color="dmax", log_x=True,
    title=f"{sample_name}: cluster size [core+border atoms]",
).show()

# how close each cluster's nearest neighbour is
pgp.plot_distribution_grid(
    grid_clusters, value="closest_cluster_distance", row="order", color="dmax",
    title=f"{sample_name}: nearest-neighbour cluster distance [nm]",
).show()

# ...and the same in units of the two clusters' radii: the left tail crossing 1
# is where clusters start overlapping, i.e. where a dmax is over-merging
pgp.plot_distribution_grid(
    grid_clusters, value="closest_cluster_distance_norm",
    row="order", color="dmax",
    title=f"{sample_name}: nearest-neighbour distance / (r₁ + r₂)",
).show()

## 4. Where clusters split and merge

Every cell of the grid clusters the same atoms, so consecutive cells can be
compared atom by atom instead of by cluster counts alone. That distinguishes
"5 fewer clusters because five precipitates merged into two" from "5 fewer
because five dissolved below n_min" - identical counts, different clusterings.

`track_cluster_splits` walks each line of the grid and reports, for every step:

| | |
|---|---|
| `n_split` | clusters of the earlier cell whose atoms land in 2+ clusters of the later one |
| `n_new_from_splits` | how many *extra* clusters those splits created (a cluster becoming 3 counts 2) |
| `n_merged` / `n_lost_to_merges` | the same from the other side |
| `n_dissolved` / `n_created` | clusters with no counterpart at all on the other side |
| `split_cluster_ids` | *which* clusters split, so one can be pulled up in 3D below |

The two axes do opposite things, and that is a property of the algorithm rather
than of the data: **raising dmax adds links**, so clusters merge but never
split; **raising the Order takes core points away**, so a cluster held together
by a thin neck of core atoms falls into two. The elbow is the step where either
count takes off - past it the parameter is not refining the clustering, it is
rearranging it.

In [ ]:
splits_dmax = pgp.track_cluster_splits(grid, axis="dmax", n_min=summary)
splits_order = pgp.track_cluster_splits(grid, axis="order", n_min=summary)

# raising dmax merges
pgp.plot_split_tracking(
    splits_dmax, value="n_merged",
    title=f"{sample_name}: clusters that swallowed a neighbour when dmax rose",
).show()

# the same, over the plane: each cell coloured by what happened arriving at it
pgp.plot_grid_heatmap(
    splits_dmax, value="n_new_from_splits", colorscale="Reds",
    title=f"{sample_name}: clusters that swallowed a neighbour when dmax rose",
).show()

# raising the Order splits
pgp.plot_split_tracking(
    splits_order, value="n_new_from_splits",
    title=f"{sample_name}: extra clusters from splits when Order rose",
).show()

# the same, over the plane: each cell coloured by what happened arriving at it
pgp.plot_grid_heatmap(
    splits_order, value="n_new_from_splits", colorscale="Reds",
    title=f"{sample_name}: extra clusters from splits when Order rose",
).show()

splits_order.drop(columns=["split_cluster_ids", "merged_cluster_ids"]).head(10)

## 5. Choose the parameters

Pick the cell off the map: a plateau where `optimal_n_min` and
`n_real_clusters` are stable across its neighbours, `min_cluster_distance_norm`
is comfortably above 1, and the split/merge counts around it are small.

`dbulk` and `derode` are the two distances the sweep never needed - how far a
cluster reaches into the matrix to capture bulk atoms, and how much of that
shell is stripped back off again. The old posgen workflow set both to `dmax/2`.

In [ ]:
######################### INPUT REQUIRED #########################
selected_order = 6
selected_dmax = 0.55

cell = summary.loc[
    (summary["order"] == selected_order)
    & np.isclose(summary["dmax"], selected_dmax)
].squeeze()

selected_n_min = int(cell["optimal_n_min"])

envelope_radius = selected_dmax / 2      # posgen's dbulk
d_erosion = selected_dmax / 2            # posgen's derode

cell

## 6. Individual clusters

Any `cluster_id` hovered off a plot above can be typed in here. The ids belong
to one cell of the grid - the same physical precipitate is numbered differently
at every (Order, dmax) - so the cell's own per-atom file is the one to read.

The other three show a cluster *changing*: the same neighbourhood at every
value along one axis of the map. The first overlays the dmax values, one colour
each. The last two give every value its own 3D subplot, coloured by ion, and
turning or zooming one turns them all: a cluster that **splits** as the Order
rises (two blobs where the subplot before showed one), then one that
**merges** as dmax rises (two blobs becoming one).

In [ ]:
######################### INPUT REQUIRED #########################
selected_cell = next(
    e for e in grid
    if e["order"] == selected_order and np.isclose(e["dmax"], selected_dmax)
)
cell_atoms = pd.read_parquet(selected_cell["real_clustered_atoms_path"])

cell_clusters = grid_clusters.loc[
    (grid_clusters["order"] == selected_order)
    & np.isclose(grid_clusters["dmax"], selected_dmax)
]

# the three biggest - or any id read off a plot above
clusters_to_plot = (
    cell_clusters.sort_values("size_core", ascending=False)["cluster_id"]
    .head(3).tolist()
)

# ...or the closest pair in the cell, which is what the normalised-distance
# heatmap flagged and usually the more interesting thing to look at:
#
#   crowded = cell_clusters.nsmallest(1, "closest_cluster_distance_norm")
#   clusters_to_plot = [int(crowded["cluster_id"].iloc[0]),
#                       int(crowded["closest_cluster_id_norm"].iloc[0])]

print("showing clusters:", clusters_to_plot)
_ = pgp.show_clusters_in_pos(cell_atoms, clusters_to_plot, ion_colors=ion_colors)

In [ ]:
# One cluster across the dmax axis of the map, one colour per dmax - the view
# is its own bounding box, so keep this to one or two clusters at a time.
# Every dmax draws the same atoms in the same place, so the markers are graded
# by size (low dmax large, high dmax small on top): a big dot with a small dot
# inside was clustered at both, a bare small dot was only picked up by the
# looser dmax. Click a dmax in the legend to hide it and compare two directly.

# TODO I don't see much use in it - give me good reasons to keep it or I'll remove it

dmax_axis = pgp.select_sweep(grid, order=selected_order)

pgp.compare_selected_clusters_across_param(
    dmax_axis, "dmax", chosen_value=selected_dmax,
    selected_clusters=clusters_to_plot[:1],
).show()

In [ ]:
# ...and a cluster that actually splits, taken off the tracking table. Those
# ids belong to the earlier cell of the step, so this walks the Order axis
# from there.
at_selected_dmax = splits_order.loc[
    np.isclose(splits_order["dmax"], selected_dmax) & (splits_order["n_split"] > 0)
]
splitting = (
    at_selected_dmax if len(at_selected_dmax)
    else splits_order.loc[splits_order["n_split"] > 0]
)

if len(splitting):
    step = splitting.sort_values("n_split", ascending=False).iloc[0]
    split_ids = list(step["split_cluster_ids"][:1])
    print(f"dmax {step['dmax']}, Order {step['from_value']} -> {step['to_value']}: "
          f"{step['n_split']} clusters split; showing {split_ids}")

    # spread=True: one 3D subplot per Order, n_cols to a row, all framing the
    # same box and coloured by ion. Turning or zooming one turns the rest with
    # it; the split is the Order whose subplot shows two blobs where the one
    # before it showed one
    pgp.compare_selected_clusters_across_param(
        pgp.select_sweep(grid, dmax=step["dmax"]), "order",
        chosen_value=step["from_value"], selected_clusters=split_ids,
        spread=True, n_cols=3, ion_colors=ion_colors,
    ).show()
else:
    print("nothing split along the Order axis anywhere in this grid")

In [ ]:
# ...and the dmax twin: a cluster that swallowed a neighbour when dmax rose,
# taken off the dmax tracking table. Merge ids belong to the *later* cell of
# the step - the merged cluster - so its box holds every piece it was made
# from, and the subplots before it show those pieces as separate blobs.
at_selected_order = splits_dmax.loc[
    (splits_dmax["order"] == selected_order) & (splits_dmax["n_merged"] > 0)
]
merging = (
    at_selected_order if len(at_selected_order)
    else splits_dmax.loc[splits_dmax["n_merged"] > 0]
)

if len(merging):
    step = merging.sort_values("n_merged", ascending=False).iloc[0]
    merged_ids = list(step["merged_cluster_ids"][:1])
    print(f"Order {step['order']}, dmax {step['from_value']} -> {step['to_value']}: "
          f"{step['n_merged']} clusters swallowed a neighbour; showing {merged_ids}")

    pgp.compare_selected_clusters_across_param(
        pgp.select_sweep(grid, order=step["order"]), "dmax",
        chosen_value=step["to_value"], selected_clusters=merged_ids,
        spread=True, n_cols=3, ion_colors=ion_colors,
    ).show()
else:
    print("nothing merged along the dmax axis anywhere in this grid")

## 7. The final run: the full pipeline

**This is where the envelope and erosion come back.** The map skipped them
because it only reads core+border counts, which they cannot change. Everything
below - composition, volume, volume fraction, and the alpha shapes in the next
section - depends on exactly which bulk atoms the envelope wraps into each
cluster and which of those erosion strips off again:

1. **core classification + linking** - the stage the map did, at the chosen
   `dmax` and `order`;
2. **size clip** - `N_min` applied to core+border counts *before* the envelope,
   as posgen does;
3. **envelope** - bulk-species atoms within `dbulk` of a classified core point
   join the cluster;
4. **erosion** - enveloped atoms within `derode` of untagged matrix are given
   back.

No relabelling here: the control runs existed to choose `n_min`, and that is
already decided. Then the corrections, in the order the old workflow applied
them - **Fm deconvolution first**, then the size columns, the neighbour
distances, and edge-cluster detection.

In [ ]:
ion_formula_map = pgp.build_ion_formula_map(rrng_df)
labelled_atoms = pgp.assign_rrng_to_pos(pos_df, rrng_df)


def final_cluster_analysis(dmax, order, n_min, dbulk, derode, print_log=False):
    """One full cluster search + the corrections pipeline, at one parameter set.

    Used for the main run and again for each CV01 perturbation below, so the
    sensitivity check measures the parameters changing and nothing else.
    """

    atoms = pgp.run_clustering(
        labelled_atoms, core_ions, bulk_ions,
        Dmax=dmax, order=order, N_min=n_min,
        envelope_radius=dbulk, d_erosion=derode,
        verbose=print_log,
    )

    stats = pgp.analyse_clusters(
        atoms, ion_formula_map=ion_formula_map, core_species=core_ions,
        atomic_volume=1 / atomic_density_per_nm3,
    )
    unclustered = pgp.summarise_unclustered(atoms, ion_formula_map=ion_formula_map)

    # Fm deconvolution, before anything is derived from the counts: the 29 Da
    # peak is an Fe/Ni overlap, split by natural Ni abundance
    if {"Fm", "Ni", "Fe"} <= set(stats.columns) and stats["Fm"].sum() > 0:
        stats = pgp.deconvolute_Fm_clusters(stats)
        if {"Fm", "Ni", "Fe"} <= set(unclustered.columns):
            unclustered = pgp.deconvolute_Fm_clusters(unclustered)

    stats["cluster_size"] = pgp.calculate_cluster_size(
        stats, core_ions, bulk_ions, elements_to_exclude
    )
    stats["cluster_size_with_matrix"] = pgp.calculate_cluster_size(
        stats, core_ions, bulk_ions, []
    )

    # nearest-neighbour distances (raw and radius-normalised), then edge clusters
    stats = pgp.find_knn_cluster_distance(stats)
    stats = pgp.remove_edge_clusters(
        atoms.loc[atoms["ion"].notna()], atoms, stats,
        random_state=0, print_log=print_log,
    )

    return atoms, stats.reset_index(drop=True), unclustered


clustered_atoms, cluster_stats, unclustered_stats = final_cluster_analysis(
    selected_dmax, selected_order, selected_n_min,
    envelope_radius, d_erosion, print_log=True,
)

cluster_stats.to_csv(
    f"{output_folder}{sample_name}_cluster-stats_corrected.csv", index=False
)
unclustered_stats.to_csv(
    f"{output_folder}{sample_name}_unclustered-stats_corrected.csv", index=False
)

print(f"\n{len(cluster_stats)} clusters, "
      f"{int(cluster_stats['is_edge_cluster'].sum())} touching the tip edge")
cluster_stats.head()

## 8. The final clusters as alpha shapes

The 3D scatters above show where the atoms are. This shows what the search
decided their **shape** was: one alpha shape per cluster, drawn see-through at
`opacity=0.6` so clusters behind other clusters stay visible, with the
**edge clusters in the second colour** - those are the ones the tip boundary
cuts through, whose volume and composition are only partly measured, and this
is where you see how much of the result is leaning on them.

`alpha` is in inverse nm: the probe sphere rolled over the atoms has radius
`1/alpha`. `1/dmax` is the natural starting point - a probe the size of the
linking distance. Raise it for a tighter, more concave surface; too high and it
tears holes, and where that happens the cluster's convex hull is drawn instead
(the tooltip says which was used). `alpha=0` is the convex hull everywhere.

The tooltip carries the cluster's id, size, neighbour distance and edge flag,
so anything that looks wrong here can be looked up in `cluster_stats` or
plotted atom by atom in section 6.

In [ ]:
######################### INPUT REQUIRED #########################
alpha = round(1 / selected_dmax, 2)     # probe sphere the size of dmax

pgp.plot_cluster_alpha_shapes(
    clustered_atoms, cluster_stats, alpha=alpha, opacity=0.6,
    title=(f"{sample_name}: cluster alpha shapes  -  dmax {selected_dmax}, "
           f"Order {selected_order}, n_min {selected_n_min}"),
).show()

## 9. Metrics, sensitivity, and the summary row

`get_cluster_metrics` reduces the run to the one row that goes in the
cross-tip table: volume fraction, number density (edge clusters counted as
half), tip volume, cluster counts.

The CV01 check re-runs the whole pipeline at `(dmax +- 0.05, Order +- 1)` -
which is to say, at two of the neighbouring cells of the map in section 2 -
and records the largest deviation per metric as an error estimate. If the map
showed a plateau around the chosen cell, these come out small; if it showed a
cliff, they will not.

In [ ]:
######################### INPUT REQUIRED #########################
dmax_interval = 0.05
order_interval = 1


def metrics_for(name, stats, unclustered, dmax, order, analysis_type):
    return pgp.get_cluster_metrics(
        reconstruction_name=name, reconstruction_location=output_folder,
        sample_type=sample_name, cluster_stats_df=stats,
        unclustered_stats_df=unclustered,
        core_ions=core_ions, bulk_ions=bulk_ions, dmax=dmax, order=order,
        n_min=selected_n_min, detector_efficiency=detector_efficiency,
        atomic_density_per_nm3=atomic_density_per_nm3,
        excluded_Fe="Fe" in elements_to_exclude,
        cluster_analysis_type=analysis_type,
    )


main_metrics = metrics_for(
    sample_name, cluster_stats, unclustered_stats,
    selected_dmax, selected_order, "main",
)

cv01_metrics = []
for sign in (-1, 1):
    this_dmax = round(selected_dmax + sign * dmax_interval, 2)
    this_order = selected_order + sign * order_interval

    _, cv_stats, cv_unclustered = final_cluster_analysis(
        this_dmax, this_order, selected_n_min, this_dmax / 2, this_dmax / 2,
    )
    cv01_metrics.append(metrics_for(
        f"{sample_name}_cv01_{sign}", cv_stats, cv_unclustered,
        this_dmax, this_order, "cv01",
    ))
    print(f"cv01 dmax={this_dmax} order={this_order}: {len(cv_stats)} clusters")

cluster_analysis_summary = pd.DataFrame([main_metrics])
cv01_df = pd.DataFrame(cv01_metrics)

for col in [
    "volume fraction", "number density per nm3", "clustered volume nm3",
    "number of all clusters", "number of non-edge clusters",
    "number of edge clusters",
]:
    cluster_analysis_summary[f"{col} cv01 error"] = (
        cv01_df[col] - main_metrics[col]
    ).abs().max()

cluster_analysis_summary.T

In [ ]:
cluster_analysis_summary.to_csv(
    f"{output_folder}{sample_name}_cluster_analysis_summary.csv", index=False
)

# The final run's atoms only exist in memory (no relabelling, so nothing was
# written for it) - persist them so a cluster can be looked up by id later
# without re-running the search.
clustered_atoms.to_parquet(
    f"{output_folder}{sample_name}_final_clustered_atoms.parquet", index=False
)

# The map's own per-cell atom files are what sections 4-6 read; keep them while
# you are still looking clusters up by id, and drop them when you are not:
#
#   import glob, os
#   for path in glob.glob(f"{output_folder}{sample_name}_order_*_real_clustered_atoms.parquet"):
#       os.remove(path)

print(f"Saved to {output_folder}. Done.")

## 10. Composition of the final clusters

What the clusters at the chosen parameters are made of, from the final run's
`cluster_stats` - so the envelope, erosion and Fm deconvolution are all in.
Every plot carries two references beside the clusters: **all clusters** pooled
(every cluster's atoms summed before normalising, so a big cluster weighs more
than a small one) and the **matrix** - every atom left unclustered.

- **Ternary** of `ternary_ions`. Each cluster is placed by those three elements
  alone, normalised to 100%, and sized by `cluster_size`; edge clusters are in
  the second colour, as in section 8. A cluster with none of the three has no
  place on the triangle and is left out - the title says how many.
- **Stacked columns**, one per cluster, smallest on the left, each normalised
  to 100% so every column is the same height and only the mix changes: read
  it for whether composition shifts with size. The percentages are over
  `composition_ions` - the core and bulk ions minus `elements_to_exclude`, the
  same atoms `cluster_size` counts - so each column's label is exactly the
  number of atoms it was normalised by. Hover a column for its `cluster_id`,
  to look it up in section 6.
- **The same columns with `elements_to_exclude` put back** - every core and
  bulk ion, normalised by `cluster_size_with_matrix`, the excluded elements
  stacked on top so the solutes still start from 0. The first version answers
  "what are the solutes in a cluster"; this one answers "how much of a cluster
  is solute at all". Read it mostly for the two reference columns: the matrix
  is nearly all Fe, and how far the pooled clusters sit from it is the
  enrichment the search found. A cluster whose column is mostly Fe is one the
  envelope wrapped a lot of matrix into - which `envelope_radius` and
  `d_erosion` control, so this plot moves with them far more than the
  solute-only one. The columns are ordered by this size, so the order can
  differ slightly from the plot above.

In [ ]:
######################### INPUT REQUIRED #########################
ternary_ions = ["Mn", "Ni", "Si"]

# the atoms cluster_size counts: core and bulk ions, minus elements_to_exclude
composition_ions = [
    ion for ion in core_ions + bulk_ions if ion not in elements_to_exclude
]

# ...and every core and bulk ion, what cluster_size_with_matrix counts - the
# excluded ones stacked last, on top, so the solutes still start from 0
composition_ions_with_matrix = composition_ions + [
    ion for ion in core_ions + bulk_ions if ion in elements_to_exclude
]
excluded = " + ".join(elements_to_exclude)

pgp.plot_cluster_ternary(
    cluster_stats, ions=ternary_ions, unclustered_stats=unclustered_stats,
    title=f"{sample_name}: {'-'.join(ternary_ions)} composition of each cluster",
).show()

pgp.plot_cluster_composition_bars(
    cluster_stats, ions=composition_ions, unclustered_stats=unclustered_stats,
    ion_colors=ion_colors,
    title=(f"{sample_name}: composition of each cluster without {excluded}, "
           "smallest to largest"),
).show()

pgp.plot_cluster_composition_bars(
    cluster_stats, ions=composition_ions_with_matrix,
    unclustered_stats=unclustered_stats, ion_colors=ion_colors,
    size_column="cluster_size_with_matrix",
    title=(f"{sample_name}: composition of each cluster with {excluded}, "
           "smallest to largest"),
).show()

---

### Where to go from here

- **One axis at a time.** `pgp.select_sweep(grid, order=...)` or
  `select_sweep(grid, dmax=...)` returns a slice of this map in exactly the
  shape the 1D notebook's graphs take - `optimise_n_min`,
  `plot_real_clusters_across_swept_param`,
  `plot_real_cluster_ratio_across_swept_param` and the composition plots all
  read it unchanged.
- **Checking the shortcut.** `pgp.validate_fast_grid(...)` runs the full
  pipeline alongside the fast one and compares them cell by cell. It is slow;
  run it once on a new tip, not every time.
- **Composition.** The map's clusters are core+border only, so their
  composition is a core-ion composition. The absolute one - core ions as a
  share of every atom in the cluster - comes from `cluster_stats` in section 7,
  and section 10 plots it.

Thanks for using PosgenPy!